# RMCE Thesis — Step 6: GPT-2 + LoRA Fine-tuning (Kaggle GPU)

Trains 5 traditions × 2 tokenisers (REMI, EC-REMI) = 10 models.

**Before running:** Add your dataset to this notebook via *Add Data* and set `DATASET_SLUG` below to match its name.

In [ ]:
# ── Install packages (run once, ~2 min) ─────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'miditok', 'symusic', 'pretty_midi', 'peft',
    'transformers', 'datasets', 'pandas', 'numpy', 'scipy'
])
print('Packages installed.')

In [ ]:
# ── Paths — set DATASET_SLUG to match your Kaggle dataset name ──────────────
from pathlib import Path
import torch

DATASET_SLUG = 'thesis-midi-data'   # <-- change this to your dataset slug

INPUT_ROOT = Path('/kaggle/input') / DATASET_SLUG
MIDI_ROOT  = INPUT_ROOT / 'data' / 'processed'
META_DIR   = INPUT_ROOT / 'data' / 'metadata'
CKPT_ROOT  = Path('/kaggle/working/outputs/checkpoints')
RESULTS    = Path('/kaggle/working/results')
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

# Verify GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify data
traditions = ['western_classical', 'hindustani', 'carnatic', 'irish_folk', 'turkish_makam']
for t in traditions:
    d = MIDI_ROOT / t / 'midi'
    n = len(list(d.glob('*.mid')) + list(d.glob('*.midi'))) if d.exists() else 0
    print(f'  {t}: {n} MIDI files')

In [ ]:
# ── EC-REMI Tokenizer (embedded) ─────────────────────────────────────────────
import sys, importlib, importlib.util as _ilu

# Write ec_remi.py to working directory so it can be loaded
EC_REMI_CODE = '''
from __future__ import annotations
import sys
from pathlib import Path
from typing import Optional
import numpy as np
import pretty_midi
from miditok import REMI, TokenizerConfig
from symusic import Score as _Score

CENTS_PER_COMMA_53TET = 1200.0 / 53
CENTS_PER_SHRUTI_HALF = 1200.0 / (22 * 2)
MICRO_TRADITIONS = {"hindustani", "carnatic", "turkish_makam"}
ORNAMENT_TRADITIONS = {"irish_folk", "turkish_makam"}

HINDUSTANI_RAGAS = [
    "Abhogi","Bahar","Bairagi","Basanti kedar","Bhairabi","Bhairav",
    "Bhatiyar","Bhimpalas","Bhoop","Bibhas","Bilaskhani todi",
    "Chandrakauns","Dagori","Des","Dhani","Gawti","Hindol Pancham",
    "Jait Kalyan","Jog","Jogia","Kalavati","Kalyan","Kedar","Khamaj",
    "Khat","Khokar","Kirwani","Komal rishabh asavari","Lagan Gandhar",
    "Lalat","Lalit pancham","Maajh khamaj","Madhukauns","Malkauns",
    "Marubihag","Marwa","Megh","Mishra kalingada","Mishra piloo",
    "Miya malhar","Nat Kamod","Nat bhairav","Puriya dhanashree",
    "Rageshri","Ramdasi malhar","Saraswati","Shree","Shuddh sarang",
    "Shuddha kalyan","Sohini","Todi","Triveni gauri","Yaman kalyan",
]
CARNATIC_RAGAS = [
    "Budham Aashrayami","Dharini Telusukonti","Dorakuna","Gam Ganapate",
    "Gamakakriya","Ganamuda Panam","Gange Maam Pahi","Gopi Gopala Bala",
    "Janakipathe","Kailasapathe","Kanakangaka Guha","Karuna Ela Gante",
    "Lalitha Lavanga","Neene Ballideno","Nera Nammiti","Paragu Matada",
    "RTP Andholika","Sami Dayajuda","Sarasamukhi Sakala",
    "Shloka Sri Ramachandra Shrita Parijata","Shlokham","Tamburi Mitidava",
    "Tillana","Tolinenu Jesina","Vinakayunna Della","behag","bhairavi",
    "chakravakam","chittaranjani","gambhira nata","hindolam",
    "kambhoji","karaharapriya","madhyamavati","mohanam","sankarabharanam",
    "sindhubhairavi","todi","vasanta",
]
TURKISH_MAKAMS = [
    "acem","acemasiran","acembuselik","acemkurdi","araban","arazbar",
    "asiranmaye","askefza","bendihisar","besteisfahan","bestenigar",
    "beyati","beyati_araban","beyati_arabanbuselik","beyatibuselik",
    "buselik","buselikasiran","buzurk","dilkeshaveran","dilkeside",
    "dilkusa","dilnisin","dugah","dugah_maye","evic","evicbuselik",
    "ferahfeza","gerdaniye","gerdaniyebuselik","goncairana","guldeste",
    "gulizar","gulzar","hicaz","hicaz_humayun","hicaz_uzzal",
    "hicaz_zemzeme","hicaz_zirgule","hicazasiran","hicazbuselik",
    "hicazkar","hicazkarkurdi","hisarbuselik","huseyni","huzi",
    "huzzam","huzzam_icedid","isfahanek","karcigar","kucek",
    "kurdi_gerdaniye","kurdilihicazkar","mahur","mahurbuselik",
    "maveraunnehr","muhayyer","muhayyerbuselik","muhayyerkurdi",
    "muhayyersunbule","murekkep_isfahan","mustear","neva","neva_kurdi",
    "nevabuselik","neveser","nihavendikebir","nihavendirumi","nihavent",
    "nikriz","nisabur","nisaburek","nisabureyn","pencgah","peykinesat",
    "rahatfeza","rahatulervah","rast","rasticedid","rastmaye","rehavi",
    "rengidil","revnaknuma","reyya","ruhnuvaz","ruyidilara","ruyiirak",
    "saba","sabaasiran","sababuselik","sabazemzeme","sazkar","sedaraban",
    "segah","segah_maye","sehnaz","sehnazbuselik","selmek","serefnuma",
    "sevk_icedid","sevkefza","sipihr","sivenuma","sultaniirak",
    "sultaniyegah","suzidil","suzidilara","suzinak","suzinak_zirgule",
    "tahir","tarzinevin","ussak","vecdidil","vechisehnaz","yegah",
    "yeni_cargah","zavil","zirefkend",
]
IRISH_MODES = [
    "Adorian","Amajor","Aminor","Amixolydian","Bminor",
    "Ddorian","Dmajor","Dminor","Dmixolydian",
    "Edorian","Emajor","Eminor","Gmajor","Gminor",
]
WESTERN_MODES = ["major","minor","unknown"]

class ECREMIVocabulary:
    def __init__(self, remi_vocab):
        self._base = dict(remi_vocab)
        self._ext = {}
        self._next_id = len(remi_vocab)
        self._build()
    def _add(self, token):
        if token not in self._base and token not in self._ext:
            self._ext[token] = self._next_id
            self._next_id += 1
    def _build(self):
        for r in HINDUSTANI_RAGAS: self._add(f"Modal_Raga_H_{r.replace(\' \',\'_\')}") 
        for r in CARNATIC_RAGAS:   self._add(f"Modal_Raga_C_{r.replace(\' \',\'_\')}") 
        for m in TURKISH_MAKAMS:   self._add(f"Modal_Makam_{m}")
        for m in IRISH_MODES:      self._add(f"Modal_Mode_{m}")
        for m in WESTERN_MODES:    self._add(f"Modal_WC_{m}")
        for d in (-2,-1,0,1,2):    self._add(f"MicroOffset_I_{d:+d}")
        for d in (-2,-1,0,1,2):    self._add(f"MicroOffset_T_{d:+d}")
        for o in ("Roll","Cut","Slide","Vibrato","Trill"): self._add(f"Ornament_{o}")
        self._add("Modal_Unknown")
    @property
    def full_vocab(self): return {**self._base, **self._ext}
    @property
    def total_vocab_size(self): return self._next_id
    @property
    def remi_vocab_size(self): return len(self._base)
    @property
    def ext_vocab_size(self): return len(self._ext)

def _build_pb_timeline(pm):
    import bisect
    times, cents = [], []
    for inst in pm.instruments:
        for pb in inst.pitch_bends:
            times.append(pb.time)
            cents.append(pb.pitch / 8192.0 * 200.0)
    if not times: return [], []
    paired = sorted(zip(times, cents))
    return [p[0] for p in paired], [p[1] for p in paired]

def _pitch_bend_cents_at(times, cents, onset_s):
    import bisect
    if not times: return 0.0
    idx = bisect.bisect_right(times, onset_s + 0.02) - 1
    return cents[idx] if idx >= 0 else 0.0

def _cents_to_comma_bin(cents, tradition):
    val = cents / (CENTS_PER_COMMA_53TET if tradition == "turkish_makam" else CENTS_PER_SHRUTI_HALF)
    return int(max(-2, min(2, round(val))))

def _detect_irish_ornaments(notes):
    ornaments, n, i = {}, len(notes), 0
    while i < n:
        note = notes[i]
        dur = note.end - note.start
        if i+2 < n:
            n2,n3 = notes[i+1],notes[i+2]
            if n3.start - note.start < 0.18 and abs(note.pitch - n3.pitch) <= 2:
                ornaments[i] = "Roll"; i+=1; continue
        if dur < 0.07 and i+1 < n:
            n2 = notes[i+1]
            if n2.start - note.end < 0.02 and (note.pitch - n2.pitch) in (1,2):
                ornaments[i] = "Cut"; i+=1; continue
        if dur < 0.10 and i+1 < n:
            n2 = notes[i+1]
            if n2.end - n2.start < 0.10 and n2.pitch - note.pitch in (1,-1) and i+2 < n:
                ornaments[i] = "Slide"; i+=1; continue
        i+=1
    return ornaments

def _detect_turkish_ornaments(notes):
    ornaments, n, i = {}, len(notes), 0
    while i < n:
        if i+3 < n:
            p = [notes[i+k].pitch for k in range(4)]
            if notes[i+3].start - notes[i].start < 0.30 and abs(p[0]-p[1])<=2 and p[0]==p[2] and p[1]==p[3]:
                ornaments[i] = "Trill"; i+=1; continue
        i+=1
    return ornaments

class ECREMITokenizer:
    def __init__(self):
        self._remi = REMI(TokenizerConfig())
        self._vocab = ECREMIVocabulary(self._remi.vocab)
        self._tok2id = self._vocab.full_vocab
        self._id2tok = {v:k for k,v in self._tok2id.items()}
    @property
    def vocab_size(self): return self._vocab.total_vocab_size
    @property
    def vocab(self): return dict(self._tok2id)
    def tokenize(self, midi_path, tradition, modal_label=None):
        midi_path = Path(midi_path)
        try:
            seqs = self._remi.encode(_Score(str(midi_path)))
        except Exception: return []
        if not seqs: return []
        base_tokens = seqs[0].tokens
        modal_token = self._resolve_modal_token(tradition, modal_label)
        pm, pb_times, pb_cents = None, [], []
        if tradition in MICRO_TRADITIONS or tradition in ORNAMENT_TRADITIONS:
            try:
                pm = pretty_midi.PrettyMIDI(str(midi_path))
                if tradition in MICRO_TRADITIONS:
                    pb_times, pb_cents = _build_pb_timeline(pm)
            except Exception: pm = None
        notes_sorted = []
        if pm is not None:
            all_notes = [n for inst in pm.instruments for n in inst.notes]
            notes_sorted = sorted(all_notes, key=lambda n: (n.start, n.pitch))
        ornament_map = {}
        if pm is not None and tradition == "irish_folk":
            ornament_map = _detect_irish_ornaments(notes_sorted)
        elif pm is not None and tradition == "turkish_makam":
            ornament_map = _detect_turkish_ornaments(notes_sorted)
        ec_tokens = [modal_token]
        note_idx = 0
        for tok in base_tokens:
            ec_tokens.append(tok)
            if tok.startswith("Pitch_") and note_idx < len(notes_sorted):
                note = notes_sorted[note_idx]
                if tradition in MICRO_TRADITIONS and pb_times:
                    cents = _pitch_bend_cents_at(pb_times, pb_cents, note.start)
                    dev = _cents_to_comma_bin(cents, tradition)
                    prefix = "MicroOffset_T_" if tradition == "turkish_makam" else "MicroOffset_I_"
                    ec_tokens.append(f"{prefix}{dev:+d}")
                if note_idx in ornament_map:
                    ec_tokens.append(f"Ornament_{ornament_map[note_idx]}")
                note_idx += 1
        return ec_tokens
    def encode(self, tokens):
        return [self._tok2id.get(t, self._tok2id.get("MASK_None", 0)) for t in tokens]
    def decode(self, ids):
        return [self._id2tok.get(i, "MASK_None") for i in ids]
    def _resolve_modal_token(self, tradition, label):
        if label is None: return "Modal_Unknown"
        lc = str(label).strip()
        if tradition == "hindustani":
            t = f"Modal_Raga_H_{lc.replace(\' \',\'_\')}"
            return t if t in self._tok2id else "Modal_Unknown"
        if tradition == "carnatic":
            t = f"Modal_Raga_C_{lc.replace(\' \',\'_\')}"
            return t if t in self._tok2id else "Modal_Unknown"
        if tradition == "turkish_makam":
            t = f"Modal_Makam_{lc.lower()}"
            return t if t in self._tok2id else "Modal_Unknown"
        if tradition == "irish_folk":
            t = f"Modal_Mode_{lc}"
            return t if t in self._tok2id else "Modal_Unknown"
        if tradition == "western_classical":
            key = lc.lower()
            t = "Modal_WC_minor" if "minor" in key else ("Modal_WC_major" if "major" in key else "Modal_WC_unknown")
            return t if t in self._tok2id else "Modal_Unknown"
        return "Modal_Unknown"
'''

ec_remi_path = Path('/kaggle/working/ec_remi.py')
ec_remi_path.write_text(EC_REMI_CODE)

spec = _ilu.spec_from_file_location('_ec_remi', str(ec_remi_path))
mod  = _ilu.module_from_spec(spec)
sys.modules['_ec_remi'] = mod
spec.loader.exec_module(mod)
ECREMITokenizer = mod.ECREMITokenizer
print('EC-REMI tokenizer loaded.')

In [ ]:
# ── Training setup ───────────────────────────────────────────────────────────
import warnings, json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from miditok import REMI, TokenizerConfig
from symusic import Score as SScore
from transformers import GPT2LMHeadModel, TrainingArguments, Trainer, set_seed
from peft import LoraConfig, get_peft_model, TaskType
warnings.filterwarnings('ignore')

CHUNK_SIZE = 512
STRIDE     = 256
N_EPOCHS   = 5
BATCH_SIZE = 8    # larger batch on GPU
LR         = 5e-4
LORA_R     = 8
LORA_ALPHA = 16
LORA_DROP  = 0.1
SEED       = 42
USE_FP16   = torch.cuda.is_available()

TRADITION_CONFIG = {
    'western_classical': {'label': 'Western Classical', 'meta_csv': 'maestro_selected.csv',   'label_col': None,    'modal_default': 'unknown'},
    'hindustani':        {'label': 'Hindustani',        'meta_csv': 'hindustani_tracks.csv',  'label_col': 'raga'},
    'carnatic':          {'label': 'Carnatic',          'meta_csv': 'carnatic_tracks.csv',    'label_col': 'raga'},
    'irish_folk':        {'label': 'Irish Folk',        'meta_csv': 'irish_folk_tunes.csv',   'label_col': 'mode'},
    'turkish_makam':     {'label': 'Turkish Makam',     'meta_csv': 'symbtr_selected.csv',    'label_col': 'makam'},
}

class MusicChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, idx):
        ids = torch.tensor(self.chunks[idx], dtype=torch.long)
        return {'input_ids': ids, 'labels': ids.clone()}

class FixedLenCollator:
    def __call__(self, features):
        ids = torch.stack([f['input_ids'] for f in features])
        return {'input_ids': ids, 'labels': ids.clone()}

def _chunk(ids):
    return [ids[i:i+CHUNK_SIZE] for i in range(0, len(ids)-CHUNK_SIZE+1, STRIDE)]

def _load_label_map(trad_key):
    tinfo = TRADITION_CONFIG[trad_key]
    if not tinfo.get('label_col'): return {}
    meta_path = META_DIR / tinfo['meta_csv']
    if not meta_path.exists(): return {}
    df = pd.read_csv(meta_path)
    midi_col = 'midi_path' if 'midi_path' in df.columns else 'processed_filename'
    if midi_col not in df.columns: return {}
    return {Path(str(r[midi_col])).name: (str(r[tinfo['label_col']]) if pd.notna(r[tinfo['label_col']]) else None)
            for _, r in df.iterrows()}

def build_remi_dataset(trad_key, remi_tok):
    midi_dir = MIDI_ROOT / trad_key / 'midi'
    files = sorted(list(midi_dir.glob('*.midi')) + list(midi_dir.glob('*.mid')))
    all_chunks, ok, err = [], 0, 0
    for fpath in files:
        try:
            seqs = remi_tok.encode(SScore(str(fpath)))
            if seqs: all_chunks.extend(_chunk(seqs[0].ids)); ok += 1
            else: err += 1
        except Exception: err += 1
    return MusicChunkDataset(all_chunks), {'ok': ok, 'err': err, 'chunks': len(all_chunks)}

def build_ec_remi_dataset(trad_key, ec_tok):
    midi_dir = MIDI_ROOT / trad_key / 'midi'
    files = sorted(list(midi_dir.glob('*.midi')) + list(midi_dir.glob('*.mid')))
    label_map = _load_label_map(trad_key)
    modal_default = TRADITION_CONFIG[trad_key].get('modal_default')
    all_chunks, ok, err = [], 0, 0
    for fpath in files:
        try:
            tokens = ec_tok.tokenize(fpath, tradition=trad_key, modal_label=label_map.get(fpath.name, modal_default))
            if tokens: all_chunks.extend(_chunk(ec_tok.encode(tokens))); ok += 1
            else: err += 1
        except Exception: err += 1
    return MusicChunkDataset(all_chunks), {'ok': ok, 'err': err, 'chunks': len(all_chunks)}

def create_gpt2_lora(vocab_size):
    model = GPT2LMHeadModel.from_pretrained('gpt2')
    model.resize_token_embeddings(vocab_size)
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROP, target_modules=['c_attn'], bias='none'
    )
    return get_peft_model(model, lora_cfg)

def train_one(trad_key, tokeniser_type, remi_tok=None, ec_tok=None):
    label = TRADITION_CONFIG[trad_key]['label']
    ckpt_dir = CKPT_ROOT / f'{trad_key}_{tokeniser_type}'

    # Skip if already trained
    if (ckpt_dir / 'final').exists():
        print(f'  [{label} | {tokeniser_type.upper()}] — already done, skipping.')
        return None

    print(f'\n{"="*64}')
    print(f'  {label}  |  {tokeniser_type.upper()}')
    print(f'{"="*64}')
    set_seed(SEED)

    if tokeniser_type == 'remi':
        dataset, stats = build_remi_dataset(trad_key, remi_tok)
        vocab_size = remi_tok.vocab_size
    else:
        dataset, stats = build_ec_remi_dataset(trad_key, ec_tok)
        vocab_size = ec_tok.vocab_size

    print(f'  Files: {stats["ok"]} ok, {stats["err"]} err | chunks: {stats["chunks"]}')
    if not dataset: print('  No data — skipping.'); return None

    val_n = max(1, int(0.1 * len(dataset)))
    train_ds = MusicChunkDataset(dataset.chunks[:-val_n])
    val_ds   = MusicChunkDataset(dataset.chunks[-val_n:])
    print(f'  Train: {len(train_ds)}  Val: {len(val_ds)}')

    model = create_gpt2_lora(vocab_size)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {n_train:,} / {n_total:,} ({100*n_train/n_total:.2f}%)')

    ckpt_dir.mkdir(parents=True, exist_ok=True)
    warmup   = max(10, len(train_ds) // (BATCH_SIZE * 10))
    log_steps = max(1, len(train_ds) // (BATCH_SIZE * 5))

    train_args = TrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=N_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR,
        warmup_steps=warmup,
        save_strategy='epoch',
        eval_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        logging_steps=log_steps,
        report_to='none',
        seed=SEED,
        use_cpu=False,
        fp16=USE_FP16,
    )

    trainer = Trainer(
        model=model, args=train_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=FixedLenCollator(),
    )
    result = trainer.train()
    trainer.save_model(str(ckpt_dir / 'final'))

    row = {
        'tradition': trad_key, 'tokeniser': tokeniser_type,
        'vocab_size': vocab_size, 'n_files_ok': stats['ok'],
        'n_chunks_train': len(train_ds), 'n_chunks_val': len(val_ds),
        'train_loss': round(result.training_loss, 4),
        'trainable_params': n_train, 'total_params': n_total,
    }
    with open(ckpt_dir / 'train_stats.json', 'w') as f:
        json.dump(row, f, indent=2)
    print(f'  Final loss: {result.training_loss:.4f}')
    return row

print('Training functions loaded.')

In [ ]:
# ── Run all 10 models ────────────────────────────────────────────────────────
remi_tok = REMI(TokenizerConfig())
ec_tok   = ECREMITokenizer()
print(f'REMI vocab: {remi_tok.vocab_size}  |  EC-REMI vocab: {ec_tok.vocab_size}')

all_rows = []
for trad in TRADITION_CONFIG:
    for tok_type in ['remi', 'ec_remi']:
        row = train_one(trad, tok_type, remi_tok=remi_tok, ec_tok=ec_tok)
        if row:
            all_rows.append(row)

if all_rows:
    out_df = pd.DataFrame(all_rows)
    out_df.to_csv(RESULTS / 'finetuning_stats.csv', index=False)
    print('\nAll results:')
    print(out_df.to_string(index=False))

In [ ]:
# ── Save outputs as a zip for download ──────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/checkpoints', 'zip', '/kaggle/working/outputs')
print('Saved: /kaggle/working/checkpoints.zip')